# 좋은 도구 이름과 설명

목록·ID·집계 도구의 설명서를 비교합니다.

## 준비

In [1]:
# 상위 폴더에서 환경 파일과 공개 주문 CSV를 찾습니다.
from pathlib import Path
from dotenv import load_dotenv

def find_up(name: str) -> Path | None:
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        candidate = folder / name
        if candidate.exists():
            return candidate
    return None

load_dotenv(find_up(".env"))
data_path = find_up("data/orders.csv")
assert data_path is not None, "공개 묶음의 챕터 폴더 또는 notebooks 폴더에서 실행합니다."
print("Python 실습 준비 완료")


Python 실습 준비 완료


In [2]:
# 세 도구가 함께 사용할 조회 함수를 준비합니다.
from pathlib import Path
from typing import Any

import pandas as pd


def fetch_orders(csv_path: str | Path, *, status: str | None = None) -> list[dict[str, Any]]:
    """CSV 주문을 읽고 선택한 상태의 행을 사전 목록으로 반환합니다."""
    orders = pd.read_csv(csv_path, dtype={"order_id": "string", "customer_id": "string"})
    if status is not None:
        orders = orders.loc[orders["status"] == status]
    return orders.to_dict(orient="records")


In [3]:
# 상태별 목록 도구에 선택 조건을 설명합니다.
from langchain.tools import tool
@tool
def list_orders_by_status(status: str) -> list[dict[str, Any]]:
    """주문 상태(paid, pending, cancelled)로 주문 목록을 조회합니다."""
    return fetch_orders(data_path, status=status)


In [4]:
# 한 건 조회와 금액 집계의 이름·반환 목적을 구분합니다.
@tool
def get_order_by_id(order_id: str) -> dict:
    """정확한 주문 ID로 주문 한 건의 상태와 금액을 조회합니다."""
    rows = fetch_orders(data_path)
    return next((r for r in rows if r["order_id"] == order_id), {"error": "주문을 찾지 못했습니다."})

@tool
def summarize_order_amounts(status: str) -> dict:
    """주문 상태별 건수와 총 주문 금액을 집계합니다."""
    rows = fetch_orders(data_path, status=status)
    return {"status": status, "count": len(rows), "total": sum(r["amount"] for r in rows)}


## 모델이 보는 설명

In [5]:
# 모델에게 전달할 도구 이름·설명·입력 칸을 확인합니다.
tools = [list_orders_by_status, get_order_by_id, summarize_order_amounts]
for item in tools:
    print({"name": item.name, "description": item.description, "args": item.args})
assert len({item.name for item in tools}) == 3


{'name': 'list_orders_by_status', 'description': '주문 상태(paid, pending, cancelled)로 주문 목록을 조회합니다.', 'args': {'status': {'title': 'Status', 'type': 'string'}}}
{'name': 'get_order_by_id', 'description': '정확한 주문 ID로 주문 한 건의 상태와 금액을 조회합니다.', 'args': {'order_id': {'title': 'Order Id', 'type': 'string'}}}
{'name': 'summarize_order_amounts', 'description': '주문 상태별 건수와 총 주문 금액을 집계합니다.', 'args': {'status': {'title': 'Status', 'type': 'string'}}}


## 설명서 크기

In [6]:
# 문자 수를 비교하며 토큰 사용량과 구분합니다.
import json
for item in tools:
    print(item.name, len(item.description), len(json.dumps(item.args, ensure_ascii=False)))


list_orders_by_status 46 49
get_order_by_id 33 53
summarize_order_amounts 26 49


## 실험 결과 해석

선택 정확도는 다음 절에서 실제로 측정합니다.